# Lecture 06 — APIs (Application Programming Interfaces)

PGR215 Data Collection and Analysis — Kristiania University College

## 1. Hva er en API?

**API** = Application Programming Interface — et grensesnitt som lar programmer kommunisere med hverandre.

### Hvorfor bruke APIer?
- Strukturert datatilgang (JSON/XML)
- Offisiell og pålitelig
- Rate-begrenset men lovlig
- Bedre enn web scraping for tilgjengelige tjenester

### REST API
De fleste moderne APIer er **RESTful**:
- Bruker HTTP-metoder (GET, POST, PUT, DELETE)
- Returnerer data i JSON-format
- Stateless (hver request er uavhengig)

## 2. HTTP-metoder

| Metode | Beskrivelse | Eksempel |
|--------|------------|----------|
| **GET** | Hent data | Hent værdata, brukerprofil |
| **POST** | Send/opprett data | Opprett bruker, send melding |
| **PUT** | Oppdater data | Endre brukerprofil |
| **DELETE** | Slett data | Slett en ressurs |

### HTTP Status Codes

| Kode | Betydning |
|------|-----------|
| 200 | OK — alt gikk bra |
| 201 | Created — ressurs opprettet |
| 400 | Bad Request — feil i forespørselen |
| 401 | Unauthorized — mangler autentisering |
| 403 | Forbidden — ingen tilgang |
| 404 | Not Found — ressurs finnes ikke |
| 429 | Too Many Requests — rate limit |
| 500 | Internal Server Error |

In [ ]:
import requests
import json
import pandas as pd

# Grunnleggende GET-request
print("=== Grunnleggende API-kall ===")
try:
    response = requests.get('https://httpbin.org/get', timeout=5)
    print(f"Status code: {response.status_code}")
    print(f"Content-Type: {response.headers['content-type']}")
    
    data = response.json()
    print(f"\nJSON-respons (nøkler): {list(data.keys())}")
    print(f"Origin IP: {data.get('origin', 'ukjent')}")
except requests.exceptions.RequestException as e:
    print(f"Feil: {e} (krever internett)")

## 3. JSON — API-enes dataformat

De fleste APIer returnerer **JSON** (JavaScript Object Notation):

```json
{
  "name": "Oslo",
  "country": "NO",
  "temp": 5.2,
  "conditions": ["cloudy", "rain"]
}
```

In [ ]:
# Jobbe med JSON i Python
import json

# Python dict ↔ JSON
weather_data = {
    "city": "Oslo",
    "country": "NO",
    "temp_celsius": 5.2,
    "humidity": 78,
    "conditions": ["overskyet", "lett regn"],
    "wind": {"speed": 3.5, "direction": "SW"}
}

# Dict → JSON-streng
json_str = json.dumps(weather_data, indent=2, ensure_ascii=False)
print("=== Python dict → JSON ===")
print(json_str)

# JSON-streng → Dict
parsed = json.loads(json_str)
print(f"\n=== Tilgang til data ===")
print(f"By: {parsed['city']}")
print(f"Temperatur: {parsed['temp_celsius']}°C")
print(f"Vind: {parsed['wind']['speed']} m/s fra {parsed['wind']['direction']}")
print(f"Forhold: {', '.join(parsed['conditions'])}")

## 4. requests-modulen

In [ ]:
# Query parameters
print("=== GET med query parameters ===")
try:
    params = {'q': 'python', 'sort': 'stars', 'per_page': 3}
    resp = requests.get('https://api.github.com/search/repositories', params=params, timeout=5)
    
    if resp.status_code == 200:
        data = resp.json()
        print(f"Totalt treff: {data['total_count']}")
        print(f"\nTopp 3 Python-repoer på GitHub:")
        for repo in data['items'][:3]:
            print(f"  {repo['full_name']}: {repo['stargazers_count']:,} stjerner")
    else:
        print(f"Feil: {resp.status_code}")
except Exception as e:
    print(f"Feil: {e}")

In [ ]:
# Headers og autentisering
print("=== Request med headers ===")
try:
    headers = {
        'User-Agent': 'PGR215-Student/1.0',
        'Accept': 'application/json'
    }
    resp = requests.get('https://httpbin.org/headers', headers=headers, timeout=5)
    data = resp.json()
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f"Feil: {e}")

print("\n=== API-nøkkel (eksempel) ===")
print("# Metode 1: Query parameter")
print("requests.get(url, params={'api_key': 'DIN_NØKKEL'})")
print("\n# Metode 2: Header")
print("requests.get(url, headers={'Authorization': 'Bearer DIN_NØKKEL'})")
print("\n# Metode 3: .env-fil (anbefalt!)")
print("from dotenv import load_dotenv")
print("import os")
print("load_dotenv()")
print("api_key = os.getenv('API_KEY')")

## 5. Eksempel: Offentlige APIer

### Noen kjente gratis APIer:

| API | Beskrivelse | URL |
|-----|------------|-----|
| JSONPlaceholder | Fake REST API for testing | jsonplaceholder.typicode.com |
| OpenWeatherMap | Værdata (krever gratis nøkkel) | openweathermap.org |
| GitHub API | Repository-data | api.github.com |
| REST Countries | Landinfo | restcountries.com |
| CoinGecko | Kryptovaluta-priser | api.coingecko.com |

In [ ]:
# Eksempel 1: JSONPlaceholder (ingen nøkkel nødvendig)
print("=== JSONPlaceholder API ===")
try:
    # Hent brukere
    resp = requests.get('https://jsonplaceholder.typicode.com/users', timeout=5)
    users = resp.json()
    
    df_users = pd.DataFrame(users)
    print(f"Hentet {len(df_users)} brukere")
    display(df_users[['id', 'name', 'email', 'phone']].head())
    
    # Hent poster for en bruker
    resp2 = requests.get('https://jsonplaceholder.typicode.com/posts', 
                         params={'userId': 1}, timeout=5)
    posts = resp2.json()
    print(f"\nBruker 1 har {len(posts)} poster:")
    for post in posts[:3]:
        print(f"  - {post['title'][:50]}...")
        
except Exception as e:
    print(f"Feil: {e}")

In [ ]:
# Eksempel 2: REST Countries
print("=== REST Countries API ===")
try:
    resp = requests.get('https://restcountries.com/v3.1/region/europe', timeout=5)
    countries = resp.json()
    
    nordic = []
    for c in countries:
        name = c['name']['common']
        if name in ['Norway', 'Sweden', 'Denmark', 'Finland', 'Iceland']:
            nordic.append({
                'Land': name,
                'Hovedstad': c.get('capital', ['?'])[0],
                'Befolkning': c.get('population', 0),
                'Areal_km2': c.get('area', 0),
                'Valuta': list(c.get('currencies', {}).keys())[0] if c.get('currencies') else '?'
            })
    
    df_nordic = pd.DataFrame(nordic).sort_values('Befolkning', ascending=False)
    print("Nordiske land:")
    display(df_nordic)
    
except Exception as e:
    print(f"Feil: {e}")

## 6. API-nøkler og .env

Mange APIer krever en **API-nøkkel** for autentisering.

### Best practice: Bruk .env-fil

```bash
# .env (ALDRI commit denne til git!)
OPENWEATHER_API_KEY=din_nøkkel_her
```

```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENWEATHER_API_KEY")
```

**Viktig**: Legg `.env` i `.gitignore`!

In [ ]:
# OpenWeatherMap-eksempel (krever API-nøkkel)
print("=== OpenWeatherMap API (eksempel) ===")
print()
print("# Slik bruker du OpenWeatherMap:")
print("# 1. Registrer deg på openweathermap.org (gratis)")
print("# 2. Hent din API-nøkkel")
print("# 3. Lagre i .env: OPENWEATHER_API_KEY=din_nøkkel")
print()
print("Kode-eksempel:")
print('''  
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")
city = "Oslo"
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

resp = requests.get(url)
data = resp.json()

print(f"By: {data['name']}")
print(f"Temperatur: {data['main']['temp']}°C")
print(f"Forhold: {data['weather'][0]['description']}")
''')

## 7. Feilhåndtering og rate limiting

In [ ]:
# Robust API-kall med feilhåndtering
import time

def safe_api_call(url, params=None, max_retries=3):
    """Gjør et API-kall med retry-logikk."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=10)
            
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                wait = 2 ** attempt  # Exponential backoff
                print(f"Rate limited! Venter {wait}s...")
                time.sleep(wait)
            elif response.status_code == 404:
                print(f"Ikke funnet: {url}")
                return None
            else:
                print(f"Feil {response.status_code}: {response.text[:100]}")
                return None
                
        except requests.exceptions.Timeout:
            print(f"Timeout (forsøk {attempt + 1}/{max_retries})")
        except requests.exceptions.ConnectionError:
            print(f"Tilkoblingsfeil (forsøk {attempt + 1}/{max_retries})")
    
    print("Alle forsøk feilet!")
    return None

# Test
result = safe_api_call('https://jsonplaceholder.typicode.com/posts/1')
if result:
    print(f"Tittel: {result['title']}")

## 8. POST-requests

In [ ]:
# POST-request: Send data til API
print("=== POST Request ===")
try:
    new_post = {
        'title': 'Min første API-post',
        'body': 'Dette er innholdet i posten',
        'userId': 1
    }
    
    resp = requests.post(
        'https://jsonplaceholder.typicode.com/posts',
        json=new_post,
        timeout=5
    )
    
    print(f"Status: {resp.status_code}")
    print(f"Respons:")
    print(json.dumps(resp.json(), indent=2))
    
except Exception as e:
    print(f"Feil: {e}")

## 9. Fra API til DataFrame

In [ ]:
# Hent data fra API og lag DataFrame
print("=== API → DataFrame ===")
try:
    resp = requests.get('https://jsonplaceholder.typicode.com/todos', timeout=5)
    todos = resp.json()
    
    df_todos = pd.DataFrame(todos)
    print(f"Hentet {len(df_todos)} todos")
    display(df_todos.head())
    
    # Analyse
    print(f"\nFullførte: {df_todos['completed'].sum()} / {len(df_todos)}")
    print(f"Fullføringsprosent: {df_todos['completed'].mean()*100:.1f}%")
    
    # Per bruker
    per_user = df_todos.groupby('userId')['completed'].agg(['sum', 'count'])
    per_user.columns = ['fullfort', 'totalt']
    per_user['prosent'] = (per_user['fullfort'] / per_user['totalt'] * 100).round(1)
    print("\nPer bruker:")
    display(per_user)
    
except Exception as e:
    print(f"Feil: {e}")

## Oppsummering

**Nøkkelkonsepter fra Lecture 06:**

1. API = grensesnitt for program-til-program kommunikasjon
2. REST API bruker HTTP: GET, POST, PUT, DELETE
3. Status codes: 200=OK, 401=Unauthorized, 404=Not Found, 429=Rate limit
4. `requests.get(url, params={...})` for GET-requests
5. `response.json()` for å parse JSON-respons
6. API-nøkler lagres i `.env` (aldri i koden!)
7. Feilhåndtering: try/except, retry med exponential backoff
8. `pd.DataFrame(api_data)` for å konvertere til analyse-format
9. Kjente APIer: JSONPlaceholder, OpenWeatherMap, GitHub, REST Countries